# Chuyển kịch bản bài giảng 4.1 thành audio bằng VieNeu-TTS

Notebook này đọc file kịch bản **`CSCTCH_Kịch bản Video Bài giảng_4.1.docx`**, trích xuất phần
**"Lời bình (Voice-over)"** của từng slide, rồi dùng mô hình TTS tiếng Việt mã nguồn mở
**[VieNeu-TTS](https://github.com/pnnbao97/VieNeu-TTS)** (Apache-2.0) để tổng hợp giọng đọc.

- **Giọng đọc:** miền **Bắc**, phong cách **tự nhiên** (`tu_nhien`). Giọng mặc định của model,
  `Phạm Tuyên` (nam · Bắc · tự nhiên), khớp đúng yêu cầu — notebook cũng cho nghe thử thêm 2 giọng nữ
  miền Bắc "tự nhiên" khác (`Trúc Ly`, `Đoan Trang`) để bạn chọn nếu muốn đổi giọng.
- **Kịch bản gốc** chia sẵn thành 3 Video (32 slide), mỗi slide có 1 đoạn lời bình — notebook giữ
  đúng cấu trúc đó khi sinh audio.
- **Kết quả:** 1 file `.wav` cho từng slide, + 1 file `.wav` gộp hoàn chỉnh cho mỗi Video, lưu trong
  thư mục `outputs/`.

**Quy trình:**
1. Cài đặt thư viện
2. Cấu hình đường dẫn & tham số
3. Đọc & phân tích kịch bản `.docx`
4. Khởi tạo VieNeu-TTS, nghe thử giọng miền Bắc
5. Sinh audio cho từng slide
6. Ghép audio theo từng Video
7. Lưu manifest & nghe thử kết quả

> **Yêu cầu:** Python ≥ 3.10, không bắt buộc GPU (mặc định chạy CPU qua ONNX Runtime, đủ nhanh cho
> khối lượng ~32 đoạn text của bài giảng này).


## 1. Cài đặt thư viện

Mặc định gói `vieneu` chạy **torch-free trên CPU** (ONNX Runtime) — không cần GPU. Nếu máy có GPU
NVIDIA (CUDA ≥ 12.8) và muốn tăng tốc khi sinh hàng loạt audio, bỏ comment 2 dòng cài `torch` bên dưới
*trước khi* cài `vieneu`.


In [37]:
#%pip install -q vieneu python-docx soundfile numpy tqdm

# (Tuỳ chọn) Chỉ cần nếu có GPU NVIDIA và muốn tăng tốc sinh audio hàng loạt:
# %pip install -q torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
#!pip install -q "transformers==4.57.6"


## 2. Cấu hình đường dẫn & tham số

Đổi `DOCX_PATH` nếu bạn chạy notebook từ vị trí khác với file kịch bản.


In [38]:
from pathlib import Path

# File kịch bản gốc (.docx)
DOCX_PATH = Path("CSCTCH_Kịch bản Video Bài giảng_4.2.docx")

# Thư mục lưu audio kết quả
OUTPUT_DIR = Path("outputs_4.2")

# Khoảng lặng (giây) chèn giữa các slide khi ghép audio hoàn chỉnh của một Video
SILENCE_BETWEEN_SLIDES = 0.8

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DOCX_PATH.exists(), f"Không tìm thấy file kịch bản: {DOCX_PATH.resolve()}"

print("Kịch bản:      ", DOCX_PATH.resolve())
print("Thư mục output:", OUTPUT_DIR.resolve())


Kịch bản:       D:\TTCT 2026\Audio\CSCTCH_Kịch bản Video Bài giảng_4.2.docx
Thư mục output: D:\TTCT 2026\Audio\outputs_4.2


## 3. Đọc & phân tích kịch bản `.docx`

Kịch bản chia thành nhiều mục **"VIDEO n: ..."**, mỗi Video gồm nhiều mục **"Slide n: ..."**. Các file
kịch bản trong bộ này dùng **2 định dạng khác nhau** cho phần lời bình, nên `parse_script()` tự nhận
diện theo từng Slide:

- **Định dạng A** (vd. `4.1`): `Slide n: <tiêu đề ngắn>`, sau đó có dòng riêng
  `Lời bình (Voice-over): "<nội dung đọc>"`.
- **Định dạng B** (vd. `4.2`, `4.3`): `Slide n: "<nội dung đọc>"` — lời bình viết gộp luôn trên dòng
  Slide, không có dòng "Lời bình" riêng, không có tiêu đề ngắn.

Cách nhận diện: nếu phần văn bản sau dấu `:` của dòng Slide bắt đầu bằng dấu ngoặc kép (hoặc khá dài)
thì coi đó là lời bình luôn (định dạng B); ngược lại coi là tiêu đề và chờ dòng "Lời bình" phía sau
(định dạng A). Các dòng **"Hiển thị: ..."** là ghi chú hình ảnh trên slide (không đọc thành tiếng) nên
luôn bị bỏ qua.


In [39]:
import re
from docx import Document

VIDEO_RE = re.compile(r"^VIDEO\s+(\d+)\s*:\s*(.+)$", re.IGNORECASE)
SLIDE_RE = re.compile(r"^Slide\s+(\d+)\s*:\s*(.+)$", re.IGNORECASE)

# Nếu phần sau dấu ':' của dòng Slide dài hơn ngưỡng này, coi luôn là lời bình (định dạng B)
# thay vì tiêu đề ngắn - phòng trường hợp lời bình không có dấu ngoặc kép bao quanh.
INLINE_NARRATION_MIN_LEN = 90


def _clean_narration(raw: str) -> str:
    return raw.strip().strip('"').strip()


def parse_script(docx_path: Path):
    doc = Document(str(docx_path))
    records = []
    cur_video, cur_video_title = None, None
    cur_slide, cur_slide_title = None, None

    def add_record(slide_title, narration):
        records.append({
            "video": cur_video,
            "video_title": cur_video_title,
            "slide": cur_slide,
            "slide_title": slide_title,
            "text": narration,
        })

    for p in doc.paragraphs:
        text = p.text.strip()
        if not text:
            continue

        m = VIDEO_RE.match(text)
        if m:
            cur_video, cur_video_title = int(m.group(1)), m.group(2).strip()
            continue

        m = SLIDE_RE.match(text)
        if m:
            cur_slide = int(m.group(1))
            body = m.group(2).strip()
            is_inline_narration = body.startswith('"') or len(body) > INLINE_NARRATION_MIN_LEN
            if is_inline_narration:
                # Định dạng B: lời bình viết gộp ngay trên dòng Slide, không có dòng "Lời bình" riêng
                narration = _clean_narration(body)
                cur_slide_title = narration[:60] + ("…" if len(narration) > 60 else "")
                add_record(cur_slide_title, narration)
            else:
                # Định dạng A: đây chỉ là tiêu đề ngắn, lời bình nằm ở dòng "Lời bình" phía sau
                cur_slide_title = body
            continue

        if text.startswith("Lời bình"):
            # Bỏ tiền tố "Lời bình" / "Lời bình (Voice-over):" và dấu ngoặc kép bao quanh
            narration = _clean_narration(text[text.find(":") + 1:])
            add_record(cur_slide_title, narration)

    return records


script = parse_script(DOCX_PATH)
n_videos = len(sorted(set(r["video"] for r in script)))
print(f"Đã trích xuất {len(script)} đoạn lời bình thuộc {n_videos} video.")


Đã trích xuất 33 đoạn lời bình thuộc 3 video.


In [40]:
# Kiểm tra nhanh kết quả phân tích
for r in script:
    preview = (r["text"][:70] + "…") if len(r["text"]) > 70 else r["text"]
    print(f"[V{r['video']} - Slide {r['slide']:02d}] {r['slide_title']:<45} | {len(r['text']):>4} ký tự | {preview}")


[V1 - Slide 01] Chào các em sinh viên. Chúng ta lại gặp nhau ở bài giảng tiế… |  392 ký tự | Chào các em sinh viên. Chúng ta lại gặp nhau ở bài giảng tiếp theo của…
[V1 - Slide 02] Chương 4.2 này sẽ trang bị cho các em kiến thức tổng quan về… |  294 ký tự | Chương 4.2 này sẽ trang bị cho các em kiến thức tổng quan về hệ thống …
[V1 - Slide 03] Đầu tiên, thuật ngữ Metro hay MRT (Mass Rapid Transit) có ng… |  445 ký tự | Đầu tiên, thuật ngữ Metro hay MRT (Mass Rapid Transit) có nghĩa là Hệ …
[V1 - Slide 04] Nhưng đường sắt đô thị không chỉ có Metro. Tùy thuộc vào lưu… |  381 ký tự | Nhưng đường sắt đô thị không chỉ có Metro. Tùy thuộc vào lưu lượng và …
[V1 - Slide 05] Vậy ưu điểm vận tải vượt trội của Metro là gì? Về công suất,… |  406 ký tự | Vậy ưu điểm vận tải vượt trội của Metro là gì? Về công suất, một giờ n…
[V1 - Slide 06] Nếu ta làm một phép so sánh kinh điển: Một đoàn tàu Metro 4 … |  390 ký tự | Nếu ta làm một phép so sánh kinh điển: Một đoàn tàu Metro 4 toa có thể…
[V1 - Slid

## 4. Khởi tạo VieNeu-TTS và nghe thử giọng miền Bắc

Model v3 Turbo có 7 giọng **miền Bắc (Bắc)**, mỗi giọng gắn cố định với 1 phong cách:

| Giọng | Giới tính | Phong cách |
|---|---|---|
| `Phạm Tuyên` **(mặc định)** | Nam | tự nhiên (`tu_nhien`) |
| `Minh Đức` | Nam | tin tức (`tin_tuc`) |
| `Thanh Bình` | Nam | kể chuyện (`doc_truyen`) |
| `Trúc Ly` | Nữ | tự nhiên (`tu_nhien`) |
| `Đoan Trang` | Nữ | tự nhiên (`tu_nhien`) |
| `Ngọc Linh` | Nữ | kể chuyện (`doc_truyen`) |
| `Mai Anh` | Nữ | tin tức (`tin_tuc`) |

Vì yêu cầu là **giọng tự nhiên**, ta chỉ nghe thử 3 giọng có phong cách `tu_nhien`:
`Phạm Tuyên` (nam), `Trúc Ly` (nữ), `Đoan Trang` (nữ).


In [41]:
from vieneu import Vieneu

tts = Vieneu()  # v3 Turbo, 48kHz - tự động chọn CPU (ONNX) hoặc GPU (PyTorch) nếu có
SR = tts.sample_rate
print(f"Sample rate: {SR} Hz")

print("\nDanh sách giọng có sẵn:")
for label, voice_id in tts.list_preset_voices():
    print(f" - {voice_id:<12} | {label}")


Sample rate: 48000 Hz

Danh sách giọng có sẵn:
 - Minh Đức     | Minh Đức — Nam · Bắc · Phong cách tin tức
 - Phạm Tuyên   | Phạm Tuyên — Nam · Bắc · Phong cách tự nhiên
 - Thái Sơn     | Thái Sơn — Nam · Nam · Phong cách kể chuyện
 - Xuân Vĩnh    | Xuân Vĩnh — Nam · Nam · Phong cách tự nhiên
 - Thanh Bình   | Thanh Bình — Nam · Bắc · Phong cách kể chuyện
 - Trúc Ly      | Trúc Ly — Nữ · Bắc · Phong cách tự nhiên
 - Ngọc Linh    | Ngọc Linh — Nữ · Bắc · Phong cách kể chuyện
 - Đoan Trang   | Đoan Trang — Nữ · Bắc · Phong cách tự nhiên
 - Mai Anh      | Mai Anh — Nữ · Bắc · Phong cách tin tức
 - Thục Đoan    | Thục Đoan — Nữ · Nam · Phong cách kể chuyện
 - Minh Triết   | Minh Triết — Nam · Nam · Phong cách tin tức
 - Thùy Dung    | Thùy Dung — Nữ · Nam · Phong cách tin tức
 - Quang Sơn    | Quang Sơn — Nam · Trung · Phong cách tự nhiên
 - Ngọc Trân    | Ngọc Trân — Nữ · Trung · Phong cách tự nhiên


In [ ]:
from IPython.display import Audio, display

STYLE = "tu_nhien"  # phong cách đọc: tự nhiên
sample_text = script[0]["text"]  # câu lời bình đầu tiên, dùng làm mẫu nghe thử

for voice_name in ["Phạm Tuyên", "Minh Đức", "Đoan Trang"]:
    print(f"--- Giọng: {voice_name} (Bắc, tự nhiên) ---")
    preview_audio = tts.infer(sample_text, voice=voice_name, style=STYLE)
    display(Audio(preview_audio, rate=SR))


## 5. Chọn giọng đọc chính thức

Mặc định dùng `Phạm Tuyên` (nam, Bắc, tự nhiên). Đổi thành `"Trúc Ly"` hoặc `"Đoan Trang"` ở ô dưới
nếu bạn thích giọng nữ hơn sau khi nghe thử ở bước 4.


In [42]:
VOICE = "Minh Đức"  # Nam · Bắc · tự nhiên (giọng mặc định của model)
print(f"Sẽ dùng giọng: {VOICE} | phong cách: {STYLE}")


Sẽ dùng giọng: Minh Đức | phong cách: tu_nhien


## 6. Sinh audio cho từng slide

Với mỗi đoạn lời bình, gọi `tts.infer(...)` rồi lưu thành 1 file `.wav` riêng trong
`outputs/video_<n>/slide_<n>.wav`. `infer()` tự động chia nhỏ văn bản dài thành các câu/cụm và
nối lại nên không cần tự cắt đoạn thủ công.


In [43]:
from collections import defaultdict
from tqdm.auto import tqdm

video_audio = defaultdict(list)   # video_id -> [audio_slide1, audio_slide2, ...] (giữ đúng thứ tự)
video_titles = {}

for rec in tqdm(script, desc="Đang tổng hợp giọng đọc"):
    video_dir = OUTPUT_DIR / f"video_{rec['video']}"
    video_dir.mkdir(parents=True, exist_ok=True)
    out_path = video_dir / f"slide_{rec['slide']:02d}.wav"

    audio = tts.infer(rec["text"], voice=VOICE, style=STYLE)
    tts.save(audio, out_path)

    rec["audio_path"] = str(out_path)
    rec["duration_sec"] = round(len(audio) / SR, 2)

    video_audio[rec["video"]].append(audio)
    video_titles[rec["video"]] = rec["video_title"]

print(f"\nHoàn tất! Đã sinh {len(script)} file audio slide trong '{OUTPUT_DIR}/video_*/'.")


Đang tổng hợp giọng đọc:   0%|          | 0/33 [00:00<?, ?it/s]


Hoàn tất! Đã sinh 33 file audio slide trong 'outputs_4.2/video_*/'.


## 7. Ghép audio theo từng Video

Mỗi Video (1, 2, 3) trong kịch bản tương ứng với một video bài giảng riêng — ghép các slide trong
cùng Video lại thành 1 file audio hoàn chỉnh, chèn khoảng lặng `SILENCE_BETWEEN_SLIDES` giữa các
slide để dễ nghe / dễ dựng video sau này.


In [44]:
import numpy as np


def concat_with_silence(waves, sr, silence_sec=0.8):
    silence = np.zeros(int(sr * silence_sec), dtype=np.float32)
    parts = []
    for i, w in enumerate(waves):
        parts.append(w)
        if i < len(waves) - 1:
            parts.append(silence)
    return np.concatenate(parts)


video_full_paths = {}
for vid in sorted(video_audio):
    merged = concat_with_silence(video_audio[vid], SR, SILENCE_BETWEEN_SLIDES)
    out_path = OUTPUT_DIR / f"Video_{vid}_full.wav"
    tts.save(merged, out_path)
    video_full_paths[vid] = str(out_path)
    print(f"Video {vid} — {video_titles[vid]}")
    print(f"  -> {out_path}  ({len(merged) / SR:.1f} giây, {len(video_audio[vid])} slide)")


Video 1 — ĐỊNH NGHĨA VÀ ĐẶC ĐIỂM CỦA HỆ THỐNG METRO (SLIDE 1 - 11)
  -> outputs_4.2\Video_1_full.wav  (284.4 giây, 11 slide)
Video 2 — HỆ THỐNG VẬN HÀNH VÀ CƠ SỞ HẠ TẦNG (SLIDE 12 - 22)
  -> outputs_4.2\Video_2_full.wav  (265.3 giây, 11 slide)
Video 3 — MÔ HÌNH TOD VÀ VAI TRÒ VĨ MÔ CỦA METRO (SLIDE 23 - 33)
  -> outputs_4.2\Video_3_full.wav  (247.2 giây, 11 slide)


## 8. Lưu manifest chi tiết

Lưu lại tiêu đề slide, nội dung lời bình, đường dẫn file audio và thời lượng của từng đoạn — hữu ích
khi dựng video (khớp audio với slide) hoặc khi cần tạo lại 1 đoạn cụ thể.


In [45]:
import json

manifest_path = OUTPUT_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(script, f, ensure_ascii=False, indent=2)

print("Đã lưu manifest:", manifest_path.resolve())


Đã lưu manifest: D:\TTCT 2026\Audio\outputs_4.2\manifest.json


## 9. Nghe thử kết quả

In [ ]:
for vid in sorted(video_full_paths):
    print(f"=== Video {vid}: {video_titles[vid]} ===")
    display(Audio(video_full_paths[vid]))


## Kết quả

```
outputs/
├── video_1/slide_01.wav ... slide_10.wav
├── video_2/slide_11.wav ... slide_20.wav
├── video_3/slide_21.wav ... slide_32.wav
├── Video_1_full.wav   # audio hoàn chỉnh cho Video 1 (Tổng quan và định nghĩa về không gian ngầm)
├── Video_2_full.wav   # audio hoàn chỉnh cho Video 2 (Phân loại CTN và định nghĩa hầm giao thông)
├── Video_3_full.wav   # audio hoàn chỉnh cho Video 3 (Đặc thù vận hành và hệ thống an toàn sinh tồn)
└── manifest.json      # thông tin chi tiết từng đoạn (tiêu đề, nội dung, đường dẫn, thời lượng)
```

> **Lưu ý:** VieNeu-TTS mặc định chèn watermark âm thanh không nghe được (`apply_watermark=True`) vào
> mọi audio sinh ra, để đánh dấu nguồn gốc AI-generated — không ảnh hưởng đến chất lượng nghe.


## 10. Xuất ảnh slide PowerPoint (tự khớp với kịch bản)

Notebook tự so sánh số slide giữa kịch bản (`script`) và file PowerPoint (`PPTX_PATH`):

- **Khớp 100% số lượng** (vd. `4.2`, `4.3`: 33=33, 31=31) → tự động dùng **toàn bộ** slide, không cần
  chỉnh gì.
- **Lệch số lượng** (vd. `4.1`: kịch bản 32 slide nhưng PPTX chỉ có slide khớp nội dung tới Slide 17)
  → notebook in cảnh báo và tạm dùng phần đầu khớp theo số lượng nhỏ hơn; bạn cần tự đối chiếu tiêu đề
  từng slide rồi **ghi đè `MATCHED_SLIDES`** ở ô dưới cho đúng (xem cách làm mẫu ở phần code — đây
  chính xác là tình huống đã gặp với `4.1`).

Sau khi có `MATCHED_SLIDES`, bước này xuất từng slide thành ảnh PNG độ phân giải cao (1920×1080), thử
lần lượt 3 cách theo thứ tự ưu tiên:

1. **PowerPoint COM** (`pywin32`) — chất lượng render đúng như PowerPoint thật. Chạy trong **1 thread
   riêng** vì kernel Jupyter thường đã khởi tạo COM ở chế độ MTA (do event loop asyncio/ZMQ), trong khi
   PowerPoint COM cần STA — nếu gọi trực tiếp trên thread chính sẽ gặp lỗi
   `com_error: (-2147221021, 'Operation unavailable', ...)`.
2. **LibreOffice + PyMuPDF** — nếu máy không có PowerPoint hoặc COM vẫn lỗi. Cần cài LibreOffice
   (miễn phí: https://www.libreoffice.org). Cách này chạy headless, không cần phiên desktop tương tác
   nên ổn định hơn COM khi gọi từ script/Jupyter.
3. **Xuất ảnh thủ công** — nếu cả 2 cách trên đều không dùng được. Mở file PowerPoint tương ứng →
   **File → Save As** → chọn kiểu file **"PNG Portable Network Graphics (*.png)"** → khi được hỏi
   **"Every Slide"** hay **"Only This Slide"**, chọn **Every Slide**. PowerPoint sẽ tạo một thư mục
   chứa `Slide1.PNG`, `Slide2.PNG`, ... Gán đường dẫn thư mục đó vào `MANUAL_EXPORT_DIR` ở ô dưới rồi
   chạy lại ô — notebook sẽ tự lấy đúng các slide cần dùng.

> Yêu cầu: PowerPoint hoặc LibreOffice — hoặc chỉ cần thao tác chuột thủ công như cách 3, không cần
> cài thêm gì.


In [ ]:
%pip install -q pywin32 imageio-ffmpeg pymupdf python-pptx


In [ ]:
from pptx import Presentation

PPTX_PATH = Path("CSCTCH chuong4_ 4.2.pptx")
assert PPTX_PATH.exists(), f"Không tìm thấy file PowerPoint: {PPTX_PATH.resolve()}"

n_script_slides = len(script)
n_pptx_slides = len(Presentation(str(PPTX_PATH)).slides)

# Mặc định: dùng toàn bộ slide từ 1 đến số nhỏ hơn giữa kịch bản và PPTX.
# - Nếu 2 số bằng nhau (khớp 100%, vd 4.2, 4.3) -> tự động dùng luôn toàn bộ, không cần chỉnh gì.
# - Nếu lệch nhau (vd 4.1: kịch bản 32 nhưng PPTX chỉ có slide khớp tới 17) -> notebook cảnh báo,
#   và bạn cần tự đối chiếu tiêu đề từng slide rồi GHI ĐÈ danh sách slide khớp bên dưới, ví dụ:
#   MATCHED_SLIDES = list(range(1, 18))
MATCHED_SLIDES = list(range(1, min(n_script_slides, n_pptx_slides) + 1))

if n_script_slides == n_pptx_slides:
    print(f"✅ Kịch bản và PPTX khớp số lượng ({n_script_slides} slide) — dùng toàn bộ.")
else:
    print(
        f"⚠️  Kịch bản có {n_script_slides} slide nhưng PPTX có {n_pptx_slides} slide — SỐ LƯỢNG LỆCH NHAU.\n"
        f"    Tạm dùng {len(MATCHED_SLIDES)} slide đầu (1-{len(MATCHED_SLIDES)}). Hãy tự đối chiếu tiêu đề\n"
        f"    từng slide (xem cách làm ở Bài 4.1) rồi ghi đè MATCHED_SLIDES cho đúng nếu cần."
    )

SLIDES_DIR = OUTPUT_DIR / "slide_images"
CLIPS_DIR = OUTPUT_DIR / "slide_clips"
VIDEO_DIR = OUTPUT_DIR / "video_out"
for d in (SLIDES_DIR, CLIPS_DIR, VIDEO_DIR):
    d.mkdir(parents=True, exist_ok=True)

audio_by_slide_check = {rec["slide"]: rec.get("audio_path") for rec in script}
missing_audio = [n for n in MATCHED_SLIDES if not audio_by_slide_check.get(n)]
assert not missing_audio, f"Thiếu audio cho slide: {missing_audio} — chạy lại Bước 6 trước."

print(f"Sẽ xuất video cho {len(MATCHED_SLIDES)} slide: {MATCHED_SLIDES[0]}-{MATCHED_SLIDES[-1]}")


In [51]:
def export_slides_powerpoint(pptx_path, slide_numbers, out_dir, width=1920, height=1080):
    '''Xuất slide bằng PowerPoint COM (Windows + Microsoft PowerPoint).

    Chạy trong 1 thread riêng: kernel Jupyter thường đã gọi CoInitialize ở chế độ MTA
    (do event loop asyncio/ZMQ), trong khi PowerPoint COM server cần STA. Một thread mới
    chưa từng khởi tạo COM sẽ có "apartment" sạch, tránh lỗi com_error (Operation unavailable).
    '''
    import threading
    result = {}
    box = {}

    def worker():
        import win32com.client
        import pythoncom

        pythoncom.CoInitialize()
        try:
            app = win32com.client.Dispatch("PowerPoint.Application")
            pres = app.Presentations.Open(str(Path(pptx_path).resolve()), WithWindow=False)
            try:
                for n in slide_numbers:
                    out_path = Path(out_dir) / f"slide_{n:02d}.png"
                    pres.Slides(n).Export(str(out_path), "PNG", width, height)
                    result[n] = out_path
            finally:
                pres.Close()
                app.Quit()
        except Exception as e:
            box["exc"] = e
        finally:
            pythoncom.CoUninitialize()

    t = threading.Thread(target=worker)
    t.start()
    t.join()

    if "exc" in box:
        raise box["exc"]
    return result


def export_slides_libreoffice(pptx_path, slide_numbers, out_dir, zoom=2.0):
    '''Phương án dự phòng #2: LibreOffice (pptx -> pdf) + PyMuPDF (pdf -> ảnh từng trang).'''
    import shutil
    import subprocess
    import fitz  # PyMuPDF

    soffice = shutil.which("soffice") or shutil.which("soffice.exe")
    if not soffice:
        raise RuntimeError(
            "Không tìm thấy LibreOffice (soffice) trên máy. "
            "Cài tại https://www.libreoffice.org rồi chạy lại ô này."
        )
    pdf_path = Path(out_dir) / (Path(pptx_path).stem + ".pdf")
    subprocess.run(
        [soffice, "--headless", "--convert-to", "pdf", "--outdir", str(out_dir), str(pptx_path)],
        check=True,
    )
    doc = fitz.open(str(pdf_path))
    mat = fitz.Matrix(zoom, zoom)
    paths = {}
    for n in slide_numbers:
        pix = doc[n - 1].get_pixmap(matrix=mat)
        out_path = Path(out_dir) / f"slide_{n:02d}.png"
        pix.save(str(out_path))
        paths[n] = out_path
    doc.close()
    return paths


# Nếu đã xuất ảnh THỦ CÔNG (PowerPoint > File > Save As > PNG > "Every Slide"), gán đường dẫn
# thư mục chứa các ảnh (Slide1.PNG, Slide2.PNG, ...) vào đây rồi chạy lại ô này.
#MANUAL_EXPORT_DIR = None  # vd: Path(r"C:\Users\ADMIN\Desktop\CSCTCH chuong4_ 4.1")
MANUAL_EXPORT_DIR = Path(r"D:\TTCT 2026\Audio\CSCTCH chuong4_ 4.2")

def export_slides_manual(src_dir, slide_numbers, out_dir):
    '''Phương án dự phòng #3: lấy ảnh đã xuất thủ công từ PowerPoint (Save As > PNG > Every Slide).'''
    import shutil

    src_dir = Path(src_dir)
    if not src_dir.exists():
        raise RuntimeError(f"Không tìm thấy thư mục: {src_dir}")

    paths = {}
    for n in slide_numbers:
        candidates = list(src_dir.glob(f"Slide{n}.PNG")) + list(src_dir.glob(f"Slide{n}.png")) \
            + list(src_dir.glob(f"Slide {n}.PNG")) + list(src_dir.glob(f"Slide {n}.png")) \
            + list(src_dir.glob(f"slide{n}.png"))
        if not candidates:
            raise RuntimeError(
                f"Không tìm thấy ảnh cho Slide {n} trong {src_dir} "
                f"(đã thử Slide{n}.PNG, 'Slide {n}.PNG', slide{n}.png)."
            )
        out_path = Path(out_dir) / f"slide_{n:02d}.png"
        shutil.copyfile(candidates[0], out_path)
        paths[n] = out_path
    return paths


try:
    slide_images = export_slides_powerpoint(PPTX_PATH, MATCHED_SLIDES, SLIDES_DIR)
    print(f"Đã xuất {len(slide_images)} ảnh slide bằng PowerPoint COM -> {SLIDES_DIR}")
except Exception as e1:
    print(f"PowerPoint COM không dùng được ({e1}).")
    try:
        slide_images = export_slides_libreoffice(PPTX_PATH, MATCHED_SLIDES, SLIDES_DIR)
        print(f"Đã xuất {len(slide_images)} ảnh slide bằng LibreOffice -> {SLIDES_DIR}")
    except Exception as e2:
        print(f"LibreOffice cũng không dùng được ({e2}).")
        if MANUAL_EXPORT_DIR is None:
            raise RuntimeError(
                "Cả PowerPoint COM lẫn LibreOffice đều không chạy được. Làm theo hướng dẫn 'Xuất ảnh "
                "thủ công' ở ô markdown phía trên, gán thư mục ảnh vào MANUAL_EXPORT_DIR rồi chạy lại ô này."
            )
        slide_images = export_slides_manual(MANUAL_EXPORT_DIR, MATCHED_SLIDES, SLIDES_DIR)
        print(f"Đã lấy {len(slide_images)} ảnh slide từ thư mục xuất thủ công -> {SLIDES_DIR}")


Đã xuất 17 ảnh slide bằng PowerPoint COM -> outputs_4.2\slide_images


## 11. Ghép ảnh slide + audio → video bài giảng

Mỗi slide khớp được ghép thành 1 đoạn video ngắn (ảnh tĩnh + audio lời bình tương ứng), sau đó nối các
đoạn lại theo từng Video. Dùng `ffmpeg` qua gói `imageio-ffmpeg` (tự tải sẵn binary, không cần cài
ffmpeg hệ thống).


In [52]:
import subprocess
import imageio_ffmpeg

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()


def run_ffmpeg(args):
    cmd = [FFMPEG, "-y", "-loglevel", "error"] + [str(a) for a in args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("ffmpeg lỗi:\n" + result.stderr)


def make_slide_clip(image_path, audio_path, out_path, fps=24):
    '''Ảnh tĩnh + audio -> 1 đoạn video mp4 (thời lượng = thời lượng audio).'''
    run_ffmpeg([
        "-loop", "1", "-i", image_path,
        "-i", audio_path,
        "-c:v", "libx264", "-tune", "stillimage",
        "-c:a", "aac", "-b:a", "192k",
        "-pix_fmt", "yuv420p", "-shortest", "-r", fps,
        out_path,
    ])


def concat_clips(clip_paths, out_path):
    list_file = Path(out_path).with_suffix(".txt")
    with open(list_file, "w", encoding="utf-8") as f:
        for p in clip_paths:
            f.write(f"file '{Path(p).resolve().as_posix()}'\n")
    run_ffmpeg(["-f", "concat", "-safe", "0", "-i", list_file, "-c", "copy", out_path])
    list_file.unlink()


In [53]:
audio_by_slide = {rec["slide"]: rec["audio_path"] for rec in script}

clip_paths = {}
for n in tqdm(MATCHED_SLIDES, desc="Đang ghép ảnh slide + audio"):
    clip_path = CLIPS_DIR / f"slide_{n:02d}.mp4"
    make_slide_clip(slide_images[n], audio_by_slide[n], clip_path)
    clip_paths[n] = clip_path

print(f"Đã ghép {len(clip_paths)} đoạn video slide -> {CLIPS_DIR}")


Đang ghép ảnh slide + audio:   0%|          | 0/17 [00:00<?, ?it/s]

Đã ghép 17 đoạn video slide -> outputs_4.2\slide_clips


In [ ]:
from collections import defaultdict

# Gom các slide ĐÃ xuất video (clip_paths) theo đúng Video mà chúng thuộc về trong kịch bản
# (tự suy ra từ `script`, không hardcode số slide/số video).
slides_by_video = defaultdict(list)
for rec in script:
    if rec["slide"] in clip_paths:
        slides_by_video[rec["video"]].append(rec["slide"])

all_videos = sorted(set(rec["video"] for rec in script))
video_lecture_paths = {}

for vid in sorted(slides_by_video):
    slide_nums = sorted(slides_by_video[vid])
    total_in_video = sum(1 for rec in script if rec["video"] == vid)
    is_complete = len(slide_nums) == total_in_video
    suffix = "" if is_complete else "_PARTIAL"
    out_path = VIDEO_DIR / f"Video_{vid}_lecture{suffix}.mp4"

    ordered_clips = [clip_paths[n] for n in slide_nums]
    concat_clips(ordered_clips, out_path)
    video_lecture_paths[vid] = out_path

    note = "" if is_complete else f"  (THIẾU {total_in_video - len(slide_nums)}/{total_in_video} slide — chưa có PPTX khớp)"
    print(f"Video {vid} — {video_titles[vid]}: {out_path.name}  ({len(slide_nums)}/{total_in_video} slide){note}")

skipped_videos = [v for v in all_videos if v not in slides_by_video]
if skipped_videos:
    print(f"\nVideo {skipped_videos}: CHƯA xuất được — chưa có slide PowerPoint tương ứng.")


In [55]:
from IPython.display import Video

for vid in sorted(video_lecture_paths):
    print(f"=== Video {vid}: {video_titles[vid]} ===")
    display(Video(filename=str(video_lecture_paths[vid]), embed=False, width=640))


=== Video 1: ĐỊNH NGHĨA VÀ ĐẶC ĐIỂM CỦA HỆ THỐNG METRO (SLIDE 1 - 11) ===


=== Video 2: HỆ THỐNG VẬN HÀNH VÀ CƠ SỞ HẠ TẦNG (SLIDE 12 - 22) ===


## Kết quả video bài giảng

```
outputs/
├── slide_images/               # ảnh PNG từng slide khớp (1920x1080)
├── slide_clips/                # video ngắn từng slide (ảnh + audio)
└── video_out/
    ├── Video_1_lecture.mp4          # đủ slide -> không có hậu tố
    ├── Video_2_lecture.mp4          # ...
    └── Video_3_lecture_PARTIAL.mp4  # thiếu slide -> có hậu tố _PARTIAL, xem log ở Bước 11 để biết thiếu slide nào
```

Nếu kịch bản và PPTX khớp 100% số lượng slide (như `4.2`, `4.3`), mọi Video sẽ xuất **đầy đủ** — không
file `_PARTIAL` nào cả. Nếu còn Video nào chưa có slide PowerPoint tương ứng (như Video 3 của `4.1`),
notebook sẽ báo rõ ở cuối Bước 11 (mục "CHƯA xuất được") — audio của các slide đó vẫn có sẵn trong
`outputs/video_<n>/` để dùng khi có slide phù hợp.

> **Lưu ý:** VieNeu-TTS mặc định chèn watermark âm thanh không nghe được (`apply_watermark=True`) vào
> mọi audio sinh ra, để đánh dấu nguồn gốc AI-generated — không ảnh hưởng đến chất lượng nghe.
